# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** 用户数据_清洗后.csv（使用之前生成的清洗数据）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path("../data/E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
    Path("/Users/yq/muc_training/data/E Commerce Dataset.xlsx"),
    Path("用户数据_清洗后.csv"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到数据文件，请修改 DATA_PATH。")

if DATA_PATH.suffix == ".xlsx":
    df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
else:
    df = pd.read_csv(DATA_PATH)
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [ ]:
df.info()

# 答案：
# 1. 一行数据代表一个电商平台用户的记录
# 2. CustomerID 是用户唯一标识
# 3. Churn 字段可作为用户流失分析的目标变量（Yes/No 表示是否流失）

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [ ]:
# TODO：创建 missing_report
# 提示：df.isna().sum() 统计缺失数量；df.isna().mean() 计算缺失比例。

missing_report = pd.DataFrame({
    "缺失数量": df.isna().sum(),
    "缺失比例": (df.isna().mean() * 100).round(2)
})
missing_report = missing_report.sort_values("缺失数量", ascending=False)

display(missing_report)

### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [ ]:
# TODO：完成两项重复值统计
duplicate_rows = df.duplicated().sum()
duplicate_customer_ids = df["CustomerID"].duplicated().sum()

print("完全重复行数：", duplicate_rows)
print("CustomerID 重复数量：", duplicate_customer_ids)

# 思考：用户 ID 重复时，不能直接删除，因为可能是同一用户的不同记录或历史快照，
# 需要先确认重复原因，可能需要保留最新记录或合并多条记录的信息。

---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [ ]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

# TODO：循环填充每列的中位数
for col in numeric_missing_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: 中位数 = {median_val:.2f}, 已填充")

# TODO：输出上述字段剩余的缺失值数量
print("\n填充后各字段剩余缺失值数量：")
print(df[numeric_missing_cols].isna().sum())

### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [ ]:
# 场景：数据缺失不是随机缺失（MNAR），例如收入字段中低收入人群更不愿意填写
# 处理思路：使用多重插补（Multiple Imputation）或根据其他相关字段（如消费行为）进行预测填充，
# 或者将缺失值作为一个单独的类别处理，以保留缺失信息本身的含义。

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [ ]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [ ]:
# TODO：完成类别标准化
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace(
    {"Phone": "Mobile Phone", "Mobile": "Mobile Phone"}
)

df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace(
    {"COD": "Cash on Delivery", "CC": "Credit Card"}
)

df["PreferedOrderCat"] = df["PreferedOrderCat"].replace(
    {"Mobile": "Mobile Phone"}
)

# TODO：重新检查类别频数
print("标准化后类别频数：")
for col in category_cols:
    print(f"\n{col}")
    print(df[col].value_counts())

---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [ ]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })


### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [ ]:
# TODO：调用函数检查三个字段
print("WarehouseToHome:")
display(iqr_outlier_summary(df["WarehouseToHome"]))

print("\nOrderCount:")
display(iqr_outlier_summary(df["OrderCount"]))

print("\nCashbackAmount:")
display(iqr_outlier_summary(df["CashbackAmount"]))

# 结论：候选异常值不能直接删除，因为 IQR 方法只是统计意义上的异常值检测，
# 这些值可能是真实存在的极端情况（如高价值用户、偏远地区用户），
# 需要结合业务知识逐一核查，确认是数据错误后再处理。

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [ ]:
# TODO：完成业务规则检查
rules = {
    "使用时长小于 0": (df["HourSpendOnApp"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}
pd.Series(rules)

---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [ ]:
# TODO：完成验收
assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

print("数据清洗验收通过。")

### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [ ]:
OUTPUT_PATH = Path("../output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# TODO：导出为 UTF-8-SIG 编码的 CSV 文件
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"已导出：{OUTPUT_PATH.resolve()}")

---
## 7. 提交前自查

- [x] 已完成缺失值报告；
- [x] 已完成重复记录检查；
- [x] 已填补指定数值字段的缺失值；
- [x] 已统一登录设备、支付方式和订单品类；
- [x] 已完成候选异常值与业务规则检查；
- [x] 已导出 ecommerce_customer_cleaned.csv；
- [x] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？

In [ ]:
# 可以作为特征的字段：
# - Tenure（在网时长）：通常在网时间越长流失概率越低
# - PreferredLoginDevice（登录设备）：不同设备用户行为特征不同
# - CityTier（城市等级）：不同城市用户消费习惯和流失倾向可能不同
# - WarehouseToHome（仓库到家距离）：距离可能影响用户满意度
# - PreferredPaymentMode（支付方式）：不同支付方式用户特征不同
# - HourSpendOnApp（使用时长）：使用越频繁的用户越不容易流失
# - NumberOfDeviceRegistered（注册设备数）：设备越多可能粘性越高
# - SatisfactionScore（满意度评分）：满意度直接影响流失
# - Complain（是否投诉）：投诉用户流失风险更高
# - CouponUsed（优惠券使用数）：优惠券使用情况反映用户活跃度
# - OrderCount（订单数）：订单越多用户粘性越高
# - DaySinceLastOrder（距上次订单天数）：时间越长流失风险越高
# - CashbackAmount（返现金额）：返现可能影响用户留存

# CustomerID 不应该用于建模，因为它只是用户的唯一标识，
# 不包含任何与流失相关的业务信息，属于无意义的特征，
# 使用它会导致模型过拟合且无法泛化到新用户。